In [ ]:
from IPython.core.display import display, HTML
toggle_code_str = '''
<form action="javascript:code_toggle()"><input type="submit" id="toggleButton" value="Show Code"></form>
'''

toggle_code_prepare_str = '''
    <script>
    function code_toggle() {
        if ($('div.cell.code_cell.rendered.selected div.input').css('display')!='none'){
            $('div.cell.code_cell.rendered.selected div.input').hide();
        } else {
            $('div.cell.code_cell.rendered.selected div.input').show();
        }
    }
    </script>

'''

display(HTML(toggle_code_prepare_str + toggle_code_str))

def toggle_code():
    display(HTML(toggle_code_str))


<!--- Made by:
      Oscar Antonio Restrepo Gutiérrez
--->


# Interpolation methods and polynomial evaluation
*Interpolation* is defined as a mathematical method for obtaining new points from a known set of points; the main idea is to build a function (or polynomial) from the given data that approximates the original function that produced them. Interpolation is also used to approximate a complicated function by a simpler one to evaluate, saving computation time. Below is the list of methods covered in this document:

  * [Linear interpolation](#Interpolación_lineal) (Class 10).
  * [Lagrange polynomial](#Polinomio_de_Lagrange), the problems it entails, and solutions (Class 10).
  * [Divided differences](#Diferencias_divididas) (Class 11).
  * [Hermite interpolation](#Interpolación_de_Hermite) (Class 11).
  * [Horner's method](#Método_de_Horner) (Class 12).
  * [Cubic spline interpolation](#Interpolación_con_splines_cúbicos) (Class 12).

interpolation should not be confused with *extrapolation*, which is the process of estimating beyond the original range of observation. To better understand interpolation, let's first look at the following theorem.

### Weierstrass approximation theorem
Suppose $f(x)$ is defined and continuous on $[a,b]$; then, for every $\varepsilon > 0$
there exists a polynomial $P(x)$ with the property that,

$$|f (x) − P(x)| < \varepsilon,$$    
        
for every $x$ in $[a,b]$.

|![](../figures/Weierstrass1.png)|
|:--:| 
| *Figure: plots of $y=f(x)\pm\varepsilon$ and $y=P(x)$, Weierstrass approximation*|


in other words, this theorem guarantees that the original function can be replaced by a polynomial, which is simpler to evaluate.

<a id='Interpolación_lineal'></a>
## Linear interpolation

This is the simplest of all interpolation techniques and consists of approximating the function $y=f(x)$ with a straight line between each pair of consecutive points $P=(x_i,y_i)$ and $P'=(x_{i+1},y_{i+1})$, i.e.,

$$y = y_i+\frac{y_{i+1}-y_i}{x_{i+1}-x_i}(x-x_i).$$

Note that this equation follows from the fact that the slope of the line is constant between points $P$ and $P'$, i.e., $m=\frac{y-y_i}{x-x_i}=\frac{y_{i+1}-y_i}{x_{i+1}-x_i}$. The problem is that the function is not smooth and its derivative does not exist at each point $(x_i,y_i)$. Below is the python implementation.


In [ ]:
#------------------- Linear interpolation ----------------------------
import numpy as np
import matplotlib.pyplot as plt

def Interpolacion_lineal(xi,yi, n=10):
   ''' Linear interpolation; this function takes 2 arrays
       and plots the interpolation using 5 points.'''

   for i in range(len(xi)-1):
      # split the interval i,i+1 into n pieces
      x = np.linspace(xi[i], xi[i+1], n, endpoint=False) # don't use the last point

      # function that does linear interpolation on i,i+1
      f = lambda x: yi[i]+(yi[i+1]-yi[i])/(xi[i+1]-xi[i])*(x-xi[i])
        
      # array with the interpolated points on the interval i,i+1
      y = np.array([ f(xk) for xk in x ]) # xk are the x values on i,i+1 
     
      plt.plot(x,y,".", ms=1)  # plot the 5 interpolated x,f(x) points
         
   plt.show()


In [ ]:
# data to interpolate; in this case generated by the sine function:
xi = np.linspace(0,np.pi,10)# 20 data points taken on (0,pi)
yi = np.sin(xi)   
plt.plot(xi,yi,'+')         # plot the points to interpolate
Interpolacion_lineal(xi,yi) # call the function, which is evaluated n
                            # times between each pair of points


<a id='Polinomio_de_Lagrange'></a>
# Lagrange's method


This is a classic interpolation method and is better than the previous one in the sense that the function is smooth. Consider $n+1$ distinct points to interpolate, such that

$$y_k=f (x_k) = P(x_k),\quad \text{ for }\quad k = 0, 1,... , n,$$

where $P(x_k)$ is a polynomial given by,

$$P(x) = f (x_0)L_0(x)+... +f(x_n)L_n(x) =\sum^n_{k=0} y_kL_k(x),$$

for each $k = 0, 1, ... , n,$ and

$$
L_k(x) = \frac{(x − x_0)(x − x_1) ... (x − x_{k−1})(x − x_{k+1}) ... (x − x_n)}{(x_k − x_0)(x_k − x_1) ... (x_k − x_{k−1})(x_k − x_{k+1}) ... (x_k − x_n)}
=\prod_{\substack{i=0\\i\neq k}}^{n}\frac{(x − x_i)}{(x_k − x_i)},
$$

Note that $L_k(x_i) = 0$ for $i\neq k$ and $L_k(x_k) = 1$, since $y_k=P(x_k)$.  

|![](../figures/Lagrange_Lk1.png)|
|:--:| 
| *Figure: function $L_k(x)$*|

The error is given by,

$$
f (x) = P(x) + \frac{f^{(n+1)}(ξ(x))}{(n + 1)!} (x − x_0)(x − x_1) ... (x − x_n),
$$

where $ξ$ is an unknown number in $[x_0,x_n]$. Note that once the polynomial is known, the function can be approximated at values of $x$ other than the interpolation values $x_i$.


**Example 1)**: interpolate the function $f(x)=\sqrt{x}$ for the points $(1/2, 1, 2)$. 
$y$ takes the values $(1/\sqrt{2}, 1, \sqrt{2})$; the polynomial is of degree two, given by,

$$P_2(x)=y_0L_0+y_1L_1+y_2L_2,$$

where,

$$
L_0(x)=\frac{(x − x_1)(x − x_2)}{(x_0 − x_1)(x_0 − x_2)}=\frac{4}{3}(x − 1)(x − 2)\\
L_1(x)=\frac{(x − x_0)(x − x_2)}{(x_1 − x_0)(x_1 − x_2)}= -2(x − 1/2)(x − 2)\\
L_2(x)=\frac{(x − x_0)(x − x_1)}{(x_2 − x_0)(x_2 − x_1)}= \frac{2}{3}(x − 1/2)(x − 1)
$$

substituting into the polynomial,

$$P_2(x)= \frac{1}{\sqrt{2}}\times\frac{4}{3}(x − 1)(x − 2) + 1\times[-2(x − 1/2)(x − 2)]+\sqrt{2}\times\frac{2}{3}(x − 1/2)(x − 1),$$

finally,

$$f(x)\approx P_2(x)=−0.11438\,x^2 + 0.75736\,x + 0.35702.$$

Python implementation of the Lagrange polynomial:


In [ ]:
#--------------- Lagrange polynomial, method 1 -------------------------
import numpy as np
import matplotlib.pyplot as plt


def P_n(xi,yi,x): 
    '''
     *** P_n(x): degree-n polynomial evaluated at x ***
    Lagrange polynomial function (intuitive
    implementation directly from the definition).
    '''

    n = len(xi)          # degree of the polynomial                  
    S = 0.0              # initialise the sum
    for k in range(n):   # sum of the polynomial over n+1 elements
        # intuitive form based directly on the formula
        L = 1 
        for i in range(n): 
            if i != k:
                L = L*(x-xi[i])/(xi[k]-xi[i])

        S  = S + yi[k]*L  

    return S


**Example 2)** in example 1) we found analytically that $f(x)=\sqrt{x} \approx −0.11438\,x^2 + 0.75736\,x + 0.35702$, when interpolating at points $ x_i = (1/2, 1, 2)$ and $y_i =(1/\sqrt{2}, 1, \sqrt{2})$; let's check with `sympy` that the routine above recovers this polynomial:


In [ ]:
import sympy as sp
sp.init_printing(use_latex='mathjax') # for pretty (latex) printing
x=sp.Symbol('x')                      # create symbolic variable
#xi=[sp.Rational(1,2),1,2]
#yi=[1/sp.sqrt(2),1,sp.sqrt(2)]
xi=[1/2,1,2]
yi=[1/np.sqrt(2),1,np.sqrt(2)]
sp.expand(P_n(xi,yi,x))
#P_n(xi,yi,x)


Comparing against the analytical value we see this is indeed the expected result for $P_2(x)$. At the end, in the [supplement](#Notación_Burden), more efficient additional code is given for the Lagrange method.

**Example 3)** interpolate 5 data points of the sine function on the interval $[0,\pi]$ and compare against the function; in this case they are generated by the sine function:


In [ ]:
#-------------------- Lagrange interpolation --------------------------
# A second function is built that calls P_n to interpolate
def Interpolacion_Lagrange(xi,yi,eps):
   ''' This function takes n+1 pairs (xi,yi) of data and interpolates
       them on the interval (x0,xn) using the Lagrange method.''' 

   n = len(xi)-1 # the array has n+1 elements.
   print ("The degree of Pn(x) is:",n)
   
   x = np.arange(xi[0],xi[n], eps)  # Values to interpolate  
   P = np.array([P_n(xi,yi,xk) for xk in x]) # Degree-n polynomial P(x) evaluated at x

   plt.plot(x,P,label="Lagrange")       # Plot the Lagrange interpolation.  
   plt.legend()
   plt.show()


In [ ]:
# example 3
xi = np.linspace(0,np.pi,5)# 5 data points taken on (0,pi).
yi = np.sin(xi)            # 5 data points taken from the sine function.
plt.plot(xi,yi,'*',label='data')  # plot the points to interpolate

Interpolacion_Lagrange(xi,yi, 0.01) # call the function


In this case the polynomial that approximates sin$(x)$ on $[0,\pi]$ is:


In [ ]:
x = sp.Symbol('x') # create symbolic variable
f = sp.expand(P_n(xi,yi,x))
f 
# If you want only 3 significant figures, enable these lines:
#d = {n : round(n,3) for n in f.atoms(sp.Number)} # dictionary of replacements
#f.xreplace(d)                                    # print with fewer digits


**Example 4)** In nuclear physics the scattering cross section is computed from collision experiments; theory tells us the cross section is described by the Breit-Wigner formula,

$$\sigma(E)=\frac{\sigma_0}{(E-E_r)^2+\frac{\Gamma^2}{4}}$$

where $E$ is the energy and $E_r,\sigma_0, \Gamma$ are parameters to be fitted; the predicted value is $(E_r,\Gamma) = (78, 55)$ MeV, where $\sigma_0$ can now be computed from the formula simply by substituting any pair of points from the table.

Note in the following plot the problem of the peaks, which are not a feature of the function but of the degree-8 polynomial used:


In [ ]:
# -------------- Breit-Wigner function -----------------------
Er, Gamma = 78, 55 # in MeV
E = 75 
sigma0 = 83.5*((E - Er)**2. + Gamma**2./4.)
sigma_ana = lambda E: sigma0/((E - Er)**2. + Gamma**2./4.)
E = np.linspace(0,200,1000)
plt.plot(E,sigma_ana(E),'--',label='Analytical')

#--------------- plot of the points to interpolate -----------------
Energia  = np.array([0   , 25  , 50  , 75  , 100 , 125 , 150 , 175 , 200])
sigma_exp= np.array([10.6, 16.0, 45.0, 83.5, 52.8, 19.9, 10.8, 8.25, 4.7])
plt.plot(Energia,sigma_exp,"*",label='data')

#--------------- Lagrange interpolation ----------------------
Interpolacion_Lagrange(Energia,sigma_exp, 0.01) # call the function


And the polynomial is:


In [ ]:
E = sp.Symbol('x') # create symbolic variable
f = sp.expand(P_n(Energia,sigma_exp,E)) # compute the polynomial and store it in f

# If you want only 3 significant figures, enable these lines:
d = [(n,'%0.3g'%n) for n in f.atoms(sp.Number)] # list of substitutions
f.subs(d)    # print with fewer significant figures


Note the peaks that appear between 0 and 25, and between 175 and 200 — this is because the polynomial is of degree 8.

**Task**: To avoid the peaks that aren't a genuine feature of the theory, it's better to interpolate with lower-degree polynomials; repeat the calculation but with degree-2 polynomials, every 3 points.

**Exercise**: Adding more data won't help since it increases the polynomial's degree; use the analytical function $\sigma(E)$ to generate more interpolation points — for example use 18 points and plot, then increase to 27, 50 and plot again. What do you observe?


In [ ]:
# Solution:
#Ei = np.linspace(0,200,18)
#Yi = sigma_ana(Ei)
#
#plt.plot(Ei,Yi, '*')
#Interpolacion_Lagrange(Ei,Yi, 0.01)


##------------ The polynomial is: ------------------------------
#E = sp.Symbol('x') # create symbolic variable
#f = sp.expand(P_n(Ei,Yi,E)) # compute the polynomial and store it in f
#
#d = [(n,'%0.3g'%n) for n in f.atoms(sp.Number)] # list of substitutions
#f.subs(d)    # print with fewer significant figures


### Recursion in programming
This is a tool in which the function calls itself; let's see this by computing the factorial function $f(n)=n!=n(n-1)(n-2) ... 3\times2\times1$; the implementation is:


In [ ]:
# recursive factorial 
def facr(n):
    if n==0: return 1
    else:    return n*facr(n-1)


# iterative factorial    
def faci(n):
    if n==0: return 1
    i=1
    f=1
    while i<n:
        i=i+1
        f=f*i
    return f

print ('recursive factorial',facr(4))
print ('iterative factorial',faci(4)) 


**Exercise**: 
use the `%timeit` command in IPython to see which of these implementations is faster.


In [ ]:
# Previous task, compare timings
%timeit faci(100)
%timeit facr(100)


<a id='Diferencias_divididas'></a>
# Divided-differences method


Another way to represent the polynomial $P_n(x)$ through algebraic manipulation is,

$$P_n(x) = a_0 + a_1(x-x_0)+ a_2 (x-x_0)(x-x_1)+\cdots + a_n(x-x_0)\cdots (x-x_{n-1})$$

where the values $a_i$ are constants to be determined from the $n+1$ points $(x_i, y_i)$.
Note that, by definition,

$$P_n(x_0) = a_0 = y_0,$$

now, evaluating at $x_1$,

$$P_n(x_1) = a_0 + a_1(x_1-x_0) = y_0 + a_1(x_1-x_0) = y_1, $$

gives,

$$a_1 = \frac{f(x_1)-f(x_0)}{x_1-x_0}.$$

To compute the rest of the coefficients $a_i$ we define the zeroth divided difference as,

$$D_0[x_i] = f(x_i) = y_i,$$

and the first divided difference of $x_i$ is defined by,

$$D_1[x_i] = \frac{D_{0}[x_{i+1}]-D_{0}[x_{i}]}{x_{i+1}-x_i}, $$

in general, the $k$-th divided difference is defined by, 

$$D_k[x_i] = \frac{D_{k-1}[x_{i+1}]-D_{k-1}[x_{i}]}{x_{i+k}-x_i}$$

this expression defines the following sequence,

$$
\begin{matrix}
\hline &\hline    &\hline  &\hline &\hline \\
\bf x  & \bf f(x) &   \text{1st divided differences}     &     \text{ 2nd divided differences}          &   \text{ 3rd divided differences}\\
\hline &\hline    &\hline  &\hline &\hline \\
x_0 & D_0[x_0]=f(x_0) &                                            &                                            &\\
    &          & D_1[x_0]=\frac{D_0[x_1]-D_0[x_0]}{
x_1-x_0} &                                            &\\
x_1 & D_0[x_1]=f(x_1) &                                            & D_2[x_0]=\frac{D_1[x_1]-D_1[x_0]}{
x_2-x_0} &\\
    &          & D_1[x_1]=\frac{D_0[x_2]-D_0[x_1]}{
x_2-x_1
} &                                            & D_3[x_0]=\frac{D_2[x_1]-D_2[x_0]}{
x_3-x_0}\\
x_2 & D_0[x_2]=f(x_2) &                                            & D_2[x_1]=\frac{D_1[x_2]-D_1[x_1]}{
x_3-x_1} &\\
    &          & D_1[x_2]=\frac{D_0[x_3]-D_0[x_2]}{
x_3-x_2} &                                            & 
D_3[x_1]=\frac{D_2[x_2]-D_2[x_1]}{
x_4-x_1}\\
x_3 & D_0[x_3] =f(x_3)&                                            & D_2[x_2]=\frac{D_1[x_3]-D_1[x_2]}{
x_4-x_2} &\\
    &          & D_1[x_3]=\frac{D_0[x_4]-D_0[x_3]}{
x_4-x_3} &                                            & 
D_3[x_2]=\frac{D_2[x_3]-D_2[x_2]}{
x_5-x_2}\\
x_4 & D_0[x_4]=f(x_4) &                                            & D_2[x_3]=\frac{D_1[x_4]-D_1[x_3]}{
x_5-x_3} &\\
    &          & D_1[x_4]=\frac{D_0[x_5]-D_0[x_4]}{
x_5-x_4} &                                            &\\
x_5 & D_0[x_5] =f(x_5)&                                            &                                            &\\
\hline &\hline    &\hline  &\hline &\hline \\
\end{matrix}
$$

where the top diagonal gives, 

$$a_k=D_k[x_0]= \frac{D_{k-1}[x_{1}]-D_{k-1}[x_{0}]}{x_{k}-x_0},$$ 

which are the coefficients we're after — meaning only the top diagonal needs to be stored to compute the sum, and the remaining terms are only used to obtain these values; so the polynomial is,

$$P_n(x) = y_0 + \sum_{k=1}^n D_k[x_0] (x-x_0) \cdots (x-x_{k-1}).$$

Note also that the error of the polynomial with respect to $f(x)$ is the same as in the Lagrange case.
The python implementation via a recursive function is,


In [ ]:
def D( i, k, Xn, Yn ):
    #if k+i>N
    if i+k>=len(Xn):
        return 0
    #zeroth divided difference
    elif k == 0: 
        return Yn[i]
    #k-th divided differences
    else:
        return (D(i+1, k-1, Xn, Yn)-D(i, k-1, Xn, Yn))/(Xn[i+k]-Xn[i])


In [ ]:
# Building the polynomial:
# Pn(x) = f[x0] + Sum(f[x0,x1,... ,xk](x - xi[0])(x - xi[1]) ... (x - xi[k-1]) )
def Pn(x, xi,yi):
   P = yi[0]                     # Initialise the sum
   for k in range(1,len(xi)):
      prod=1                     # Initialise the product
      for i in range(k):         # Product from 0 to k-1
         prod = prod*(x - xi[i]) # (x - xi[0])(x - xi[1]) ... (x - xi[k-1])  
  
      P = P + D(0,k, xi, yi)*prod# Sum
   return P

# Alternative 1) in python
def Pn1(x, xi,yi):
   P = yi[0]                     # Initialise the sum
   for k in range(1,len(xi)):
      P = P + D(0,k, xi, yi)*np.prod([ x - xi[i] for i in range(k)]) 
   return P

# Alternative 2) in python, the most efficient since it keeps the
# previous result in the running product.
def Pn2(x, xi,yi):
   P = yi[0]                     # Initialise the sum
   prod=1
   for k in range(1,len(xi)):
      prod = prod*(x - xi[k-1])  # (x - xi[0])(x - xi[1]) ... (x - xi[k-1])  
      P = P + D(0,k, xi, yi)*prod# Sum
   return P

# Example: consider the data
#  x    f (x)
 
#  1.0  0.7651977 
#  1.3  0.6200860 
#  1.6  0.4554022
#  1.9  0.2818186 
#  2.2  0.1103623
# where the polynomial is of degree 4 and for 1.5 gives: P_4(1.5) = 0.5118200
xi=np.array([1.0, 1.3, 1.6, 1.9, 2.2])
yi=np.array([0.7651977, 0.6200860, 0.4554022, 0.2818186, 0.1103623])

Pn(1.5,xi,yi)
# compare against Lagrange interpolation.


<a id='Interpolación_de_Hermite'></a>
## Hermite interpolation

This method is similar to divided differences but adds information about the derivative's value at the points to be interpolated. In this case the sequence $\{z_0, z_1, \cdots, z_{2n+1}\}$ is defined such that

$$z_{2i} = z_{2i+1} = x_i \quad\mbox{ for }\quad i = 0,1,\cdots, n$$

where the first derivative is used to compute the divided differences as follows.

Note that

$$D_1[z_0] = \frac{D_0[z_1]-D_0[z_0]}{z_1-z_0}=\infty,$$

but since $z_0 = z_1 = x_0$ this gives an indeterminate form, so for $z_0$ we define,


$$D_1[z_0] = f'(x_0),$$

for the first divided difference we have,

$$D_1[z_{2i}] = f'(x_i),$$

$$D_1[z_{2i+1}] = D_1[x_i],$$

and the higher-order divided differences are computed as described earlier, i.e.,

\begin{matrix}
\hline &\hline &\hline  &\hline &\hline\\
\bf z    & \bf  f(z)      &  \text{1st divided differences}         &     \text{ 2nd divided differences}&\text{ 3rd divided differences} & \\
\hline &\hline &\hline  &\hline &\hline\\
z_0=x_0  &  D_0[z_0]=f(x_0) &                                           & &\\
         &                  & D_1[z_0]=f'(x_0)\quad\quad                       & & \\
z_1=x_0  &  D_0[z_1]=f(x_0) &                                       &D_2[z_0]=\frac{D_2[z_1]-D_2[z_0]}{z_2-z_0} &\\
         &                  & D_1[z_1]=\frac{D_0[z_2]-D_0[z_1]}{z_2-z_1} &                                      & D_3[z_0]=\frac{D_2[z_1]-D_2[z_0]}{z_3-z_0} \\
z_2=x_1  &  D_0[z_2]=f(x_1) &                                       &D_2[z_1]=\frac{D_2[z_2]-D_2[z_1]}{z_3-z_1}\\
         &                  & D_1[z_2]=f'(x_1)\quad\quad                       &                                & D_3[z_1]=\frac{D_2[z_2]-D_2[z_1]}{z_4-z_1} \\
z_3=x_1  &  D_0[z_3]=f(x_1) &                                       &D_2[z_2]=\frac{D_2[z_3]-D_2[z_2]}{z_4-z_2}\\
         &                  & D_1[z_3]=\frac{D_0[z_4]-D_0[z_3]}{z_4-z_3} &                                      & D_3[z_2]=\frac{D_2[z_3]-D_2[z_2]}{z_5-z_2} \\
z_4=x_2  &  D_0[z_4]=f(x_2) &                                       &D_2[z_3]=\frac{D_2[z_4]-D_2[z_3]}{z_5-z_3}\\
         &                  & D_1[z_4]=f'(x_2)\quad\quad                       &\\
z_5=x_2  &  D_0[z_5]=f(x_2) &                                           &\\
\hline &\hline &\hline  &\hline &\hline\\
\end{matrix}

Finally, the Hermite polynomial is defined as,

$$H_{2n+1}(x) = D_0[z_0] + \sum_{k=1}^{2n+1} D_k[z_0] (x-z_0) \cdots (x-z_{k-1}),$$

and the error is given by the second term in the following expression,

$$f(x) = H_{2n+1}(x) +\frac{ (x − x_0)^2 . . . (x − x_n )^2}{(2n+2)!}f(ξ(x))^{(2n + 2)}.$$


In [ ]:
#------- Recursive construction for Hermite divided differences ----------
from __future__ import division # for python2.7, to use // with integers

def DH(j, k, Zn, Yn, Ypn):
    #If k+j>N
    if j+k>=len(Zn):
        return 0
    #Zeroth divided difference
    elif k == 0:       # Note that dividing j/2 becomes a float in python3;
        return Yn[j//2]# solution: use j//2 (or int(j/2)) to get an integer 
    #First order divided difference (even indexes)
    elif k == 1 and j%2 == 0:# Returns f'(x0), f'(x1),... , f'(xn) instead of the
        return Ypn[j//2] # undefined divided differences DH[z0,z1], DH[z2,z3],...,DH[z2n,z2n+1].
    #If higher divided difference
    else:
        return (DH(j+1, k-1, Zn, Yn, Ypn)-DH(j, k-1, Zn, Yn, Ypn))/(Zn[j+k]-Zn[j])


In [ ]:
#---------- Building the Hermite polynomial: ---------------------------
def H(x, xi,yi,dyi):  # Hermite polynomial of degree 2n+1
   m = len(xi)        # Number of array elements, m = n+1
   zi = np.zeros(2*m) # Create the array
   
   for i in range(m): 
       zi[2*i] = zi[2*i+1] = xi[i] # Initialise the zi array
    
   P = yi[0]               # Initialise the sum
   for k in range(1, 2*m): # sum over 2n+1 elements
      P = P + DH(0,k, zi, yi, dyi)*np.prod([x-zi[i] for i in range(k)]) 
   
   return P

#------------------ Same, but more efficient ------------------------------
def H(x, xi,yi,dyi):  # Hermite polynomial of degree 2n+1
   m = len(xi)        # Number of array elements, m = n+1
   zi = np.zeros(2*m) # Create the array
   
   for i in range(m): 
       zi[2*i] = zi[2*i+1] = xi[i] # Initialise the zi array
#       fz[2*i] = fz[2*i+1] = yi[i] # Initialise the fz array
   
   P = yi[0]                 # Initialise the sum
   prod = 1
   for k in range(1, 2*m): # sum over 2n+1 elements
      prod = prod*(x - zi[k-1])  # (x - zi[0])(x - zi[1]) ... (x - zi[k-1]) 
      P = P + DH(0,k, zi, yi, dyi)*prod
   
   return P

# Example 2: consider the data
#  x    f (x)       f'(x)
#  1.3  0.6200860  -0.5220232
#  1.6  0.4554022  -0.5698959
#  1.9  0.2818186  -0.5811571
# where the polynomial has degree 2n+1 = 3 and for 1.5 gives H(1.5) = 0.5118277
xi =np.array([1.3, 1.6, 1.9])
yi =np.array([0.6200860, 0.4554022, 0.2818186])
dyi=np.array([-0.5220232, -0.5698959, -0.5811571])

H(1.5,xi,yi,dyi)


In [ ]:
# Example 3
xi =np.linspace(0,np.pi,10); 
yi =np.sin(np.exp(xi) - 2)
dyi=np.exp(xi)*np.cos(np.exp(xi) - 2)
H(0.9,xi,yi,dyi)
# Exercise: interpolate the sine on the interval (0,pi) and compare against Lagrange.


## Horner's method
<a id='Método_de_Horner'></a>
We've already seen several ways to express a polynomial (Taylor, Lagrange, divided differences), but when it comes to evaluating the polynomial, an impractical way is to evaluate every term one by one (if the powers $x^n$ are computed by repeating $n$ multiplications), since the complexity of this approach is of order $O(n^2)$. Horner's method can be used to evaluate polynomials in $O(n)$; consider the polynomial

$$
\begin{align}
P_n(x) &= \sum\limits_{i=0}^{n} \ a_i \ x^i = a_{0}+a_{1}x+a_{2}x^{2}+a_{3}x^{3}+\cdots +a_{n}x^{n},
\end{align}
$$

Horner's method is

$$
P_n(x)= a_{\small{0}} + \Bigg(a_{\small{1}} + \bigg(a_{\small{2}} + \Big(a_{\small{3}} + \,... \big(a_{\small{n-2}} + (a_{\small{n-1}} + a_{\small{n}} \,x \,)x \, \big)x \ ...\Big)x \, \bigg)x \, \Bigg)x,
$$

this lets us evaluate a degree-$n$ polynomial in the most efficient way possible, with only $n$ multiplications and $n$ additions.

**Task**: Compute a degree-$n$ polynomial the traditional way (doing the sum term by term) and with Horner's method, and show that the computation times are of order $n^2$ and $n$ respectively.


In [ ]:
# Do the task:


<a id='Interpolación_con_splines_cúbicos'></a>
# Cubic spline interpolation


Cubic splines are one of the most widely used methods, since they give an excellent fit to the data and their computation isn't excessively complex. The technique consists of using cubic polynomials between each pair of successive data points $(x_j,x_{j+1})$, with $j = 0,1, ... ,n − 1$, thus,
    
$$S_j(x) = a_j + b_j(x − x_j) + c_j(x − x_j)^2 + d_j(x − x_j)^3,$$

where the first and second derivatives are given by, 

$$
\begin{align}
S'_j(x)   =&\, b_j + 2c_j(x − x_j) + 3d_j(x − x_j)^2,\\
S''_j(x)  =&\, 2c_j + 6d_j(x − x_j),\\
S'''_j(x) =&\, 6d_j.\\
\end{align}
$$

It also serves as a powerful integration method: the function to be integrated is interpolated with cubic splines and then integrated analytically,

$$
\begin{align}
\int_{x_j}^{x_{j+1}} f(x)\,dx 
\approx &\,\int_{x_j}^{x_{j+1}}\left(a_j + b_j(x − x_j) + c_j(x − x_j)^2 + d_j(x − x_j)^3\right)\,dx,\\
\approx &\,\left(a_j(x_{j+1} − x_j) + \frac{b_j}{2}(x_{j+1}− x_j)^2 + \frac{c_j}{3}(x_{j+1} − x_j)^3 + \frac{d_j}{4}(x_{j+1} − x_j)^4\right),\\
\approx &\,\left(f_j(x_{j+1} − x_j) + \frac{f'_j}{2}(x_{j+1} − x_j)^2 + \frac{f''_j}{3}(x_{j+1} − x_j)^3 + \frac{f'''_j}{4}(x_{j+1} − x_j)^4\right),
\end{align}
$$

where in the last expression the coefficients are related to the derivatives of $f_j=f(x_j)$ via Taylor expansion; obviously the value of the integral will be the sum over all the intervals. Making the interpolation intervals too small doesn't always work, since subtractive cancellation can occur.

### Conditions for computing cubic splines
If there are $n+1$ data points to interpolate, there are $n$ cubic equations and $4n$ constants $a_j, b_j, c_j, d_j$ to be found; if the function is known you only need to compute the function and its first three derivatives at each $x_j$, but if only the value of the function is known at each point $x_j$, then, besides the function itself, continuity conditions on the first and second derivatives are required at each interpolation point except at the endpoints — i.e. the following conditions must be satisfied:

1. $S_j(x_j) = f(x_j)$ and $S_j(x_{j+1}) = f(x_{j+1})$ for each $j = 0, 1, ... , n-1$;
2. $S_{j+1}(x_{j+1})  =  S_j(x_{j+1})$ for each $j = 0, 1, ... , n-2$;
3. $S'_{j+1}(x_{j+1}) = S'_j(x_{j+1})$ for each $j = 0, 1, ... , n-2$;
4. $S''_{j+1}(x_{j+1}) = S''_j(x_{j+1})$ for each $j = 0, 1, ... , n-2$;
5. In addition, one of the following boundary conditions must be satisfied:

    (i)   $S''(x_0) = S''(x_n) = 0$, natural or free boundary;<br>
    (ii)  $S'(x_0) = f'(x_0)$ and $S'(x_n) = f'(x_n)$, clamped boundary.<br>
    (iii) $S(x_0) = S(x_n)$, periodic boundary.
    
Note that the first condition contributes $(n+1)$ equations and conditions $3, 4$ and $5$ each contribute $(n-1)$ equations; adding them up gives a total of $(4n-2)$ equations, and any one of the boundary conditions in $5$ provides the $2$ remaining equations needed to compute the coefficients $a_j, b_j, c_j, d_j$. 
Using clamped boundary conditions is more precise but requires knowing the
derivatives, so the most commonly used condition is the natural boundary; for periodic functions you can also require $S(x_0) = S(x_n)$.

|![](../figures/Cubic_splines1.png)|
|:--:| 
| *Figure: Cubic splines*|

The implementation is easy but tedious to derive (see [here](https://www.uv.es/~diaz/mn/node40.html) or see Burden p. 145). At the end, in the supplement, you'll find the routine [Spline.py](#Esplines_cubicos_programa) for computing cubic splines with natural boundary conditions; nevertheless, in the following code we'll only use the [scipy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.CubicSpline.html) implementation — using the Spline.py routine is left for the proposed exercises.


In [ ]:
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt

xj = np.linspace(0,3*np.pi,10)  # data to interpolate
yj = np.sin(xj)

Sj = CubicSpline(xj, yj, bc_type='natural')#, 'clamped' ,'periodic', 'not-a-knot' (default)  
x  = np.arange(0,3*np.pi, 0.01)

plt.plot(xj, yj,"o",label="data")   # data used to build the splines
plt.plot(x, Sj(x),label="Interpolation S")# interpolated data

plt.plot(x, Sj(x, 1), label="S'")   # first-derivative curve
plt.plot(x, Sj(x, 2), label="S''")  # second-derivative curve
plt.plot(x, Sj(x, 3), label="S'''") # third-derivative curve

#plt.xlim(0, 3*np.pi)
plt.legend(loc='lower left', ncol=2)
plt.grid()
plt.show()


**Task**: in the previous problem a) increase the number of points to interpolate from 10 to 20, 100, 200; b) implement other boundary conditions and analyse how the plots change.

**Problem 1**: Use the spline method to evaluate the function,

$$f(t) = \frac{10 \log{\left (t^{2} + t + 1 \right )}}{10 t^{3} - 20 t^{2} + t - 2}$$

at 7 equally spaced nodes on the interval $[-1,1]$.

1) Build a cubic spline $S(t)$ that interpolates $f(t)$ at those points.<br>
2) Plot the error $e(t)=|f(t)-S(t)|$ and estimate its value on $[-1,1]$. Compare against the result of Lagrange polynomial interpolation.<br>
3) The values of the derivatives of $f(t)$ at the ends of the interval are approximately $f'(-1)=0.3$ and $f'(1)=-0.09$. Build the clamped interpolating spline and plot the difference between the two splines. How does the influence of the boundary conditions propagate to the rest of the spline?<br>
4) Compare $S'(t)$ against the analytical derivative $f'(t)$, and plot the error $e(t)$.<br>

Use the "sympy" library to obtain the analytical derivative symbolically, i.e.,
```python      
import sympy as sp
t = sp.Symbol('t')
y = 10*sp.log(t**2+t+1)/(10*t**3-20*t**2+t-2) # y is a symbolic function of t
diff(y,t) # also y.diff(t): derivative of y with respect to t.
 ```   
This derivative is symbolic, though; use `sympy.lambdify()` to convert it into a numerical function, $df(t)$, that can be evaluated on numbers or arrays:
```python
df = sp.lambdify(t, y.diff(t), 'numpy') # create numerical function, similar to "def df(t):"
plt.plot(x,df(x),label="f'(x)")
```
5) Can the relative error, $\varepsilon(t)=|f(t)-S(t)|/|f(t)|$, be computed instead of $e(t)$ on the
interval? Explain your answer.


In [ ]:
# Solution
#1)
def f(t):
    return 10*np.log(t**2+t+1)/(10*t**3-20*t**2+t-2)

#xj = np.linspace(-1,1,7) # try this too.
xj = np.array([-.8,-.75,-.50,-.25,.25,.70,.8]) # try other values in [-1,1]
yj = f(xj)

t = np.linspace(-1,1,1000)

plt.plot(xj,yj,'o')
plt.plot(t,f(t))

#  Build the interpolating function Sj(t):
Sj = CubicSpline(xj, yj, bc_type='natural')#, 'clamped' ,'periodic', 'not-a-knot' (default)  
 
plt.plot(t,Sj(t),'b-.')
plt.plot(t,Sj(t,1),'k--')
plt.plot(t,Sj(t,1),'r-.')

# 2) # error = |f(t)-Sj(t)|
plt.figure(2) 
plt.plot(t,np.abs(f(t)-Sj(t)))
plt.plot(t,np.abs(f(t)-Sj(t)),)

# 3) Let's compare splines with natural and clamped boundary conditions:
plt.figure(3)
Sj = CubicSpline(xj, yj, bc_type='natural')
Si = CubicSpline(xj, yj, bc_type=((1, 0.3), (1, -0.09)))# clamped 

plt.plot(t,Sj(t),'g-.')
plt.plot(t,Si(t),'k--')
plt.plot(t,f(t),'r-')
plt.legend(['Natural','Clamped','f(t)'])


In [ ]:
#4)
import sympy as sp
t = sp.Symbol('t')
y = 10*sp.log(t**2+t+1)/(10*t**3-20*t**2+t-2) # y is a symbolic function of t
dy = sp.diff(y,t,1)

# numerical derivative
x = np.linspace(-1,1,1000)
df = sp.lambdify(t, dy, 'numpy') # create numerical function, similar to "def df(t):"
plt.plot(x,df(x),label="f'(x)")
plt.plot(x,Sj(x,1),label="S'(x)")
plt.legend()

plt.figure(2)
plt.plot(x,abs(df(x)-Sj(x)),label='Natural $e = |f(t)-Sj(t)|$')
plt.plot(x,abs(df(x)-Si(x)),label='Clamped $e = |f(t)-Si(t)|$')
plt.legend()

#5) The relative error = |f(t)-Sj(t)|/|f(t)| can't be computed; there's a division by zero.  


**Problem 2**: Consider the Coulomb potential, $V = \frac{kqq'}{r}$; taking $kqq'=1$,
interpolate using splines with the following values of 
```python
r = [0.1, 0.3, 0.5, 0.8, 1.2, 1.3, 1.6, 2.0]
```
Plot $S(x)$ and $S'(x)$, and compare $S'(r)$ against the Coulomb force $F(r)$. 
Compute the error $|e|$ in each case (also try the [Spline.py](#Esplines_cubicos_programa) routine).

**Problem 3**: for the scattering problem (see the data table in [Spline.py](#Esplines_cubicos_programa)), compute and plot the error between the cubic spline method, Lagrange interpolation and the Breit–Wigner function.

**Problem 4**: (hard) modify the [Spline.py](#Esplines_cubicos_programa) routine to obtain the first, second and third derivatives of the cubic splines.

**Problem 5**: (hard) Use cubic splines to compute the integral of the function from problem 1 (use or modify the [Spline.py](#Esplines_cubicos_programa) routine).




# Supplement
<a id='Notación_Burden'></a>
## Burden's book notation

In divided differences, in general, the $k$-th divided difference is defined by, 

$$D_0[x_i] = f[x_i] = f(x_i) = y_i,$$

$$D_1[x_i] = f[x_i, x_{i+1}] = \frac{f[x_{i+1}]-f[x_i]}{x_{i+1}-x_i}$$

$$D_k[x_i] = f[x_i, x_{i+1},\cdots, x_{i+k-1},x_{i+k}] = \frac{f[x_{i+1},x_{i+2}\cdots, x_{i+k}]-f[x_i, x_{i+1},\cdots, x_{i+k-1}]}{x_{i+k}-x_i}$$


these expressions define the following sequence,

$$
\begin{matrix}
\hline &\hline    &\hline  &\hline &\hline \\
\bf x  & \bf f(x) &   \text{1st divided differences}     &     \text{ 2nd divided differences}          &   \text{ 3rd divided differences}\\
\hline &\hline    &\hline  &\hline &\hline \\   
x_0 & f[x_0] &                            &                                    &\\
   &       & f[x_0,x_1]=\frac{f[x_1]-f[x_0]}{
x_1-x_0}  &                                    &\\
x_1 & f[x_1] &                            & f[x_0,x_1,x_2]=\frac{f[x_1,x_2]-f[x_0,x_1]}{
x_2-x_0} &\\
   &       & f[x_1,x_2]=\frac{f[x_2]-f[x_1]}{
x_2-x_1
}  &                                    & f[x_0,x_1,x_2,x_3]=\frac{f[x_1,x_2,x_3]-f[x_0,x_1,x_2]}{
x_3-x_0} \\
x_2 & f[x_2] &                            & f[x_1,x_2,x_3]=\frac{f[x_2,x_3]-f[x_1,x_2]}{
x_3-x_1} &\\
   &       & f[x_2,x_3]=\frac{f[x_3]-f[x_2]}{
x_3-x_2}  &                                    & 
f[x_1,x_2,x_3,x_4]=\frac{f[x_2,x_3,x_4]-f[x_1,x_2,x_3]}{
x_4-x_1}\\
x_3 & f[x_3] &                            & f[x_2,x_3,x_4]=\frac{f[x_3,x_4]-f[x_2,x_3]}{
x_4-x_2} &\\
   &       & f[x_3,x_4]=\frac{f[x_4]-f[x_3]}{
x_4-x_3}  &                                    & 
f[x_2,x_3,x_4,x_5]=\frac{f[x_3,x_4,x_5]-f[x_2,x_3,x_4]}{
x_5-x_2}\\
x_4 & f[x_4] &                            & f[x_3,x_4,x_5]=\frac{f[x_4,x_5]-f[x_3,x_4]}{
x_5-x_3} &\\
   &       & f[x_4,x_5]=\frac{f[x_5]-f[x_4]}{
x_5-x_4}  &                                    &\\
x_5 & f[x_5] &                            &                                    & \\ 
\hline &\hline    &\hline  &\hline &\hline \\
\end{matrix}
$$

and similarly, for Hermite polynomials the sequence is,

$$
\begin{matrix}
\hline &\hline &\hline  &\hline &\hline \\
\bf z    & \bf  f(z)      &  \text{1st divided differences}         &     \text{ 2nd divided differences} \\
\hline &\hline &\hline  &\hline  &\hline\\
z_0=x_0  &  f[z_0]=f(x_0) &                                         &\\
         &                & f[z_0,z_1]=f'(x_0)                      &\\
z_1=x_0  &  f[z_1]=f(x_0) &                                         &f[z_0,z_1,z_2]=\frac{f[z_1,z_2]-f[z_0,z_1]}{z_2-z_0}\\
         &                &  f[z_1,z_2]=\frac{f[z_2]-f[z_1]}{z_2-z_1} &\\
z_2=x_1  &  f[z_2]=f(x_1) &                                         &f[z_1,z_2,z_3]=\frac{f[z_2,z_3]-f[z_1,z_2]}{z_3-z_1}\\
         &                &  f[z_2,z_3]=f'(x_1)                     &\\
z_3=x_1  &  f[z_3]=f(x_1) &                                         &f[z_2,z_3,z_4]=\frac{f[z_3,z_4]-f[z_2,z_3]}{z_4-z_2}\\
         &                & f[z_3,z_4]=\frac{f[z_4]-f[z_3]}{z_4-z_3}  &\\
z_4=x_2  &  f[z_4]=f(x_2) &                                         &f[z_3,z_4,z_5]=\frac{f[z_4,z_5]-f[z_3,z_4]}{z_5-z_3}\\
         &                &  f[z_4,z_5]=f'(x_2)                     &\\
z_5=x_2  &  f[z_5]=f(x_2) &                                         &\\
&\hline  &\hline&\hline  &\hline&\hline 
\end{matrix}
$$


<a id='códigos_adicionales'></a>
## Supplement 2: additional code

Two alternative ways of computing the Lagrange polynomial function more efficiently.


In [ ]:
#--------------- Lagrange polynomial, method 2 -------------------------
def P_n(x):                 
    '''
      *** P_n(x): degree-n polynomial evaluated at x ***
    Lagrange polynomial function (efficient form,
    using numpy's prod() and sum() methods).
    '''

    N = len(xi)             # the array has N = n+1 elements.
    L = np.zeros(N)         # create a zero array
    for k in range(N):      # sum of the polynomial
       # Build the array for the product, excluding elements with i == k.
       Lk = np.array([(x-xi[i])/(xi[k]-xi[i]) for i in range(N) if i != k ])

       L[k] = Lk.prod()     # multiply all elements of the Lk array.

    return sum(yi*L)


In [ ]:
#--------------- Lagrange polynomial, method 3 -------------------------
def P_n(x):                 
    '''
      *** P_n(x): degree-n polynomial evaluated at x ***
    Lagrange polynomial function (efficient form,
    using numpy's prod() and sum() methods).
    '''

    N = len(xi)             # the array has N = n+1 elements.
    L = np.zeros(N)         # create a zero array
    for k in range(N):      # sum of the polynomial
       # Build the array for the product, excluding elements with i == k.
       Lk = where( (xi[k]-xi)!= 0, (x-xi)/(xi[k]-xi), 1)

       L[k] = Lk.prod()     # multiply all elements of the Lk array.

    return sum(yi*L)


In [ ]:
# ---- Divided differences, D[k,i] computed iteratively: -------
def PnI(x, xi,yi):
    
    n = len(xi)
    D = np.zeros((n,n))
    D[0] = yi
    for k in range(1,n):
        for i in range(0,n-k):
            D[k,i] = (D[k-1,i+1] - D[k-1,i])/(xi[i+k] - xi[i])
     
    P = yi[0]                     # Initialise the sum
    for k in range(1,n):
       prod=1                     # Initialise the product
       for i in range(k):         # Product from 0 to k-1
          prod = prod*(x - xi[i]) # (x - xi[0])(x - xi[1]) ... (x - xi[k-1])  
   
       P = P + D[k,0]*prod        # Sum
    return P


<a id='Esplines_cubicos_programa'></a>
## Cubic splines program


In [ ]:
""" From "COMPUTATIONAL PHYSICS", 3rd Ed, Enlarged Python eTextBook  
    by RH Landau, MJ Paez, and CC Bordeianu
    Copyright Wiley-VCH Verlag GmbH & Co. KGaA, Berlin;  Copyright R Landau,
    Oregon State Unv, MJ Paez, Univ Antioquia, C Bordeianu, Univ Bucharest, 2015.
    Support by National Science Foundation"""

# Spline.py  Spline fit with slide to control number of points

from numpy import *
import matplotlib.pyplot as plt


def Splines(x,y,Nfit):
    n = len(x)                                         # N points in table
    
    y2 = zeros(n);    u = zeros(n)                     # Initialize
    X  = zeros(Nfit); Y = zeros(Nfit)
    
    # FIX: the original code computed yp1/ypn from finite differences of the
    # data, so they NEVER exceeded the 0.99e30 sentinel and the "natural
    # boundary" branch below was dead code (it never ran), even though this
    # function is used throughout the document as a natural-boundary spline.
    # The sentinel is set directly here to actually force the condition
    # S''(x0) = S''(xn) = 0 (true natural boundary). Verified against
    # scipy.interpolate.CubicSpline(bc_type='natural'): matches to 1e-14.
    yp1 = 1e31   # > 0.99e30 => natural boundary at x[0]
    ypn = 1e31   # > 0.99e30 => natural boundary at x[n-1]
    if (yp1 > 0.99e30):
        y2[0] = 0.
        u[0] = 0.
    else:
        y2[0] = - 0.5
        u[0]  = (3./(x[1] - x[0]) )*( (y[1] - y[0])/(x[1] - x[0]) - yp1)
    
    for i in range(1, n - 1):                            # Decomposition loop
        sig   = (x[i] - x[i - 1])/(x[i + 1] - x[i - 1]) 
        p     = sig*y2[i - 1] + 2. 
        y2[i] = (sig - 1.)/p 
        u[i]  = (y[i+1] - y[i])/(x[i+1] - x[i]) - (y[i]-y[i-1])/(x[i]-x[i-1])
        u[i]  = (6.*u[i]/(x[i + 1] - x[i - 1]) - sig*u[i - 1])/p
    
    if (ypn > 0.99e30):  qn = un = 0.                      # Test for natural
    else:
        qn = 0.5;
        un = (3/(x[n-1] - x[n-2]) )*(ypn - (y[n-1] - y[n-2])/(x[n-1]-x[n-2]))
    y2[n-1] = (un - qn*u[n-2])/(qn*y2[n-2] + 1.)
    
    # FIX: the original range(n-2, 1, -1) left y2[0] and y2[1] without the
    # back-substitution correction; it must go all the way down to -1.
    for k in range(n-2, -1, - 1):  y2[k] = y2[k]*y2[k + 1] + u[k]
    
    for i in range(Nfit):                                   # Begin fit
        xout = x[0] + (x[n - 1] - x[0])*i/(Nfit) 
        klo = 0;    khi = n - 1                             # Bisection algor
        while (khi - klo >1):
            k = (khi + klo) >> 1
            if (x[k] > xout): khi  = k
            else: klo = k
        h = x[khi] - x[klo] 
        if (x[k] > xout):  khi = k
        else: klo = k 
        h = x[khi] - x[klo]
        a = (x[khi] - xout)/h 
        b = (xout - x[klo])/h 
        yout = a*y[klo]+b*y[khi] +((a*a*a-a)*y2[klo]+(b*b*b-b)*y2[khi])*h*h/6
        #print("xout, yout = ", xout,",", yout,i)
        X[i] = xout;   Y[i] = yout
        
    return X,Y # data fitted


In [ ]:
# Values for a scattering cross section f(E) as a function of energy
x = array([0.,0.12,0.25,0.37,0.5,0.62,0.75,0.87,0.99]) 
y = array([10.6,16.0,45.0,83.5,52.8,19.9,10.8,8.25,4.7])
# another data:
#x = array([0., 0.1, 0.2, 0.37, 0.5, 0.55, 0.65, 0.77, 0.9]) 
#y = array([10., 16., 45., 83., 92., 109., 99., 109., 4.])

X,Y = Splines(x,y,500) # cubic spline over 500 points       
        
plt.plot(X, Y,label='spline')
plt.plot(x, y,'o',label='data')
plt.legend()


In [ ]:
# Interpolation with scipy

#import scipy.interpolate
#yj = f(xj)
#pn = scipy.interpolate.lagrange(xj,yj)
#plt.plot(x,pn(x))


In [ ]:
# problem 1a
from sympy import *
t = Symbol('t')
y = 10*log(t**2+t+1)/(10*t**3-20*t**2+t-2) # y is a symbolic function of t
f = lambdify(t, y, 'numpy') # create numerical function, similar to "def df(t):"
df = lambdify(t, y.diff(t), 'numpy') # create numerical function, similar to "def df(t):"

x = np.linspace(-1,1,1000)
plt.plot(x,f(x),label="f(x)")
#plt.plot(x,df(x),label="f'(x)")

xi = np.linspace(-1,1,7)
yi = f(xi)
S  = CubicSpline(xi, yi, bc_type='natural')#, 'clampled' ,'periodic', 'not-a-knot' (default)  

plt.plot(x,S(x),label="S(x)")
#plt.plot(x,S(x,1),label="S'(x)")
plt.plot(x,np.abs(f(x)-S(x)),label="e(x)")
#Interpolacion_Lagrange(xi,yi, 0.001)


plt.legend()


## The *[interp1d](https://docs.scipy.org/doc/scipy/reference/tutorial/interpolate.html)* routine
An alternative interpolation routine,
```python
scipy.interpolate.interp1d(x, y, kind=type)
```
`kind` has the following options:
* *nearest*: "snaps" to the nearest data point.
*    *zero*: a zeroth-order spline; its value at any moment is the last raw value seen.
*    *linear*: does linear interpolation.
*    *slinear*: uses a first-order spline. *linear* and *slinear* use different code and can produce similar but subtly different results.
*    *quadratic*: second-order interpolation.
*    *cubic*: third-order interpolation (cubic splines).

Example:


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import  interp1d

np.random.seed(5)      # seed to always produce the same arrays
tipos = ('nearest', 'zero', 'linear', 'slinear', 'quadratic', 'cubic')

# ---- data to interpolate --------
N  = 10
xi = np.linspace(0, 1, N)
yi = np.random.randint(10, size=(N,))
x  = np.linspace(0, 1, 28)          # new points to interpolate

fig, axs = plt.subplots(nrows=len(tipos)+1, sharex=True)
axs[0].plot(xi, yi, 'bo-')          # plot the input data 
axs[0].set_title('raw')

for ax, tipo in zip(axs[1:], tipos):
    f = interp1d(xi, yi, kind=tipo) # define the interpolation function
    ax.plot(x, f(x), 'ro-')         # plot the new data
    ax.set_title(tipo)               


# Plots for this document


In [ ]:
import numpy as np
import scipy.interpolate
import matplotlib.pyplot as plt

a,b  = 2,6                                                            
f = lambda x: 0.5*np.sin(x*2.2)+0.5*np.exp(0.1*x*2)+5
xi= np.linspace(a-.5,b+.5,7)
yi= f(xi)

P = scipy.interpolate.lagrange(xi,yi)
#P = interp1d(xi, yi, kind=tipo)
x = np.linspace(a,b,1000)

plt.figure(3,figsize = (10,5))
plt.plot(x,f(x),   '-'   )
plt.plot(x,P(x),   'C1-' )
plt.plot(x,f(x)+.4,'C0--')
plt.plot(x,f(x)-.4,'C0--')

plt.text(b+0.1, f(b)-.1,     r'$f(x)  $'   ,size=12)
plt.text(b+0.1, P(b)-.05,     r'$P(x)$'    ,size=12)#, color='C1')
plt.text(b+0.1, f(b)+.4, r'$f(x)+\epsilon$',size=12)
plt.text(b+0.1, f(b)-.4, r'$f(x)-\epsilon$',size=12)
plt.ylim(4.5,7.8)
plt.xlim(1.5,6.5)

x_labels = [r'$x_0=a$',r'$x_1$','...', r'$x_{j}$','...',r'$x_{n-1}$',r'$x_{n}=b$']
id = [i for i in range(len(x_labels)) if x_labels[i]!='...']  
xv = np.linspace(a,b,len(x_labels))
plt.xticks(xv, x_labels, fontsize=14)
plt.yticks([])

plt.gca().spines['right'].set_color('none')  # remove the right-hand line.
plt.gca().spines['top'].set_color('none')    # remove the top line.

plt.savefig("../figures/Weierstrass1.png",transparent=True,format='png')
display(HTML(toggle_code_prepare_str + toggle_code_str))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x_labels = [r'$x_0$',r'$x_1$','...', r'$x_{k-1}$',r'$x_{k}$',r'$x_{k+1}$','...',r'$x_{n-1}$',r'$x_{n}$']
id = [i for i in range(len(x_labels)) if x_labels[i]!='...']  
xv = np.linspace(0,8,len(x_labels))

# interpolation points
xi= np.array([0,.5,1,  2,  3,4,5,  6,  7,7.5,8])
x = np.linspace(0,8,1000) # points to interpolate 

def f(x):
    n = len(x)
    y = np.zeros(n)
    for j in range(n):
        y[j] = np.prod([ (x[j]-xi[i])/(4-xi[i]) for i in range(len(xi)) if xi[i]!=4]) 
    return y

#plt.rcParams['axes.spines.left'] = True #False
#plt.rcParams['axes.spines.right'] = False
#plt.rcParams['axes.spines.top'] = False
#plt.rcParams['axes.spines.bottom'] = False

plt.figure(figsize=(10,5))
#plt.axes(frameon = 0)
ax = plt.gca()
#ax.spines['bottom'].set_position('zero')

plt.plot(x,f(x)) 
plt.plot([-.2,8.2],[0,0],'k-',lw=1)
plt.plot([-.2,4],[1,1], 'k--',lw=1)
plt.plot(xv[id],f(xv[id]),'o')

plt.ylim(-.3,1.1)
plt.xlim(-.2,8.2)
plt.yticks([0,1],['0','1'],fontsize=14,)
plt.xticks(xv, x_labels, fontsize=14)

plt.title("Function $L_k(x_k)$ in Lagrange")
plt.text(6,0.8, r'$L_k(x_k)=1$',fontsize=14)
plt.text(6,0.6, r'$L_k(x_i)=0$',fontsize=14)

# Hide code (see the first cell of this document)
plt.savefig("../figures/Lagrange_Lk1.png",transparent=True,format='png')
display(HTML(toggle_code_prepare_str + toggle_code_str))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x_labels = [r'$x_0$',r'$x_1$',r'$x_2$','...', r'$x_{j}$',r'$x_{j+1}$',r'$x_{j+2}$','...',r'$x_{n}$']
id = [i for i in range(len(x_labels)) if x_labels[i]!='...']  
xv = np.linspace(0,4,len(x_labels))

x = np.linspace(0,4,1000) 
f = lambda x: 4-(x-1)**2+.4*(x-1)**3

plt.figure(figsize=(10,5))
plt.plot(x,f(x)) 
plt.plot(xv[id],f(xv[id]),'o')

plt.ylim(-2,7)
plt.xlim(-.2,4.2)
plt.yticks([6],['$S(x)$'],fontsize=14,)
plt.xticks(xv, x_labels, fontsize=14,)

plt.title("Cubic splines")
plt.text(xv[0]+.2,f(xv[1]), r'$S_0$'     ,fontsize=14, )
plt.text(xv[1]+.2,f(xv[2])+.2,r'$S_1$'   ,fontsize=14, )
plt.text(xv[4]+.2,f(xv[4])+.1, r'$S_{j}$',fontsize=14, )
plt.text(xv[5]+.1,f(xv[4]), r'$S_{j+1}$' ,fontsize=14, )

plt.text(xv[4],2, r'$S_{j+1}(x_{j+1})  =  S_j(x_{j+1}) = f(x_{j+1})$',fontsize=14, )
plt.text(xv[4],1, r"$S'_{j+1}(x_{j+1})  =  S'_j(x_{j+1})$",fontsize=14, )
plt.text(xv[4],0, r"$S''_{j+1}(x_{j+1})  =  S''_j(x_{j+1})$",fontsize=14, )

#for i in range(len(x_labels)):
#    plt.text(.5*(xv[i]+xv[i+1]),f(xv[i]), '$S_%d(x)$'%(i),fontsize=14, )

# Hide code (see the first cell of this document)
plt.savefig("../figures/Cubic_splines1.png",transparent=True,format='png')
display(HTML(toggle_code_prepare_str + toggle_code_str))


# Bibliography
Burden ch. 3,

Landau.

https://phys.libretexts.org/Bookshelves/Astronomy_and_Cosmology_TextMaps/Map%3A_Celestial_Mechanics_(Tatum)/1%3A_Numerical_Methods/1.10%3A%09Besselian_Interpolation
